In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
ACCESS_TOKEN = os.getenv("CLIO_ACCESS_TOKEN")

# Keeping it simple to see everything
url = "https://eu.app.clio.com/api/v4/matters.json?fields=id,display_number,status"
headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

print("Checking for matters in Clio EU...")
response = requests.get(url, headers=headers)

print(f"Status Code: {response.status_code}")
print("Raw Response:")
print(response.text)

Checking for matters in Clio EU...
Status Code: 200
Raw Response:
{"meta":{"paging":{},"records":1},"data":[{"id":14537018,"display_number":"00001-Contact","status":"Open"}]}


In [5]:
import os, requests, json
from dotenv import load_dotenv

load_dotenv()
ACCESS_TOKEN = os.getenv("CLIO_ACCESS_TOKEN")
headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

# We force Clio to give us the nested custom_field object by adding it to the URL
url = f"https://eu.app.clio.com/api/v4/matters/14537018.json?fields=custom_field_values{{id,value,custom_field}}"

response = requests.get(url, headers=headers).json()
print(json.dumps(response.get("data", {}).get("custom_field_values", []), indent=2))

[
  {
    "id": "date-47958395",
    "value": "2021-05-05",
    "custom_field": {
      "id": 482522,
      "etag": "\"75cda69a209f16a68f6d0d2c36334927\""
    }
  },
  {
    "id": "text_line-109328603",
    "value": "JOHN GRILLO",
    "custom_field": {
      "id": 482525,
      "etag": "\"75cdd69a20a086a68f6d0d2c36334927\""
    }
  },
  {
    "id": "text_area-37622699",
    "value": "A cyclist was struck by a motor vehicle while making a left turn, with both parties reporting their respective positions.",
    "custom_field": {
      "id": 482528,
      "etag": "\"75ce069a20a1e6a68f6d0d2c36334927\""
    }
  }
]


In [8]:
import os
import requests
from dotenv import load_dotenv
import json

load_dotenv()
ACCESS_TOKEN = os.getenv("CLIO_ACCESS_TOKEN")

# Removed the strict ?fields parameter so Clio doesn't get mad!
url = "https://eu.app.clio.com/api/v4/document_templates.json"

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

print("Fetching your Document Templates from Clio EU...\n")
response = requests.get(url, headers=headers)

if response.status_code == 200:
    templates = response.json().get("data", [])
    if not templates:
        print("⚠️ No templates found. Did you upload the Word document to Clio yet?")
    else:
        print(f"✅ Found {len(templates)} templates!\n")
        
        # Sneak peek at the raw data to see the exact keys
        print("--- RAW TEMPLATE DATA (First Item) ---")
        print(json.dumps(templates[0], indent=2))
        print("--------------------------------------\n")
        
        print("🎯 YOUR TEMPLATE IDs:")
        for template in templates:
            # Safely grab filename or name, depending on what Clio uses
            display_name = template.get("filename") or template.get("name") or "Unknown Name"
            print(f"- {display_name} : {template['id']}")
else:
    print("❌ Failed to fetch templates:")
    print(response.text)

Fetching your Document Templates from Clio EU...

✅ Found 1 templates!

--- RAW TEMPLATE DATA (First Item) ---
{
  "id": 359777,
  "etag": "\"57d6169a340116a68f6d0d2c36334927\""
}
--------------------------------------

🎯 YOUR TEMPLATE IDs:
- Unknown Name : 359777


In [9]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
ACCESS_TOKEN = os.getenv("CLIO_ACCESS_TOKEN")

# We know ?fields=id,name works perfectly for Custom Fields!
url = "https://eu.app.clio.com/api/v4/custom_fields.json?fields=id,name"

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

print("Fetching all Custom Fields...\n")
response = requests.get(url, headers=headers)

if response.status_code == 200:
    fields = response.json().get("data", [])
    print(f"✅ Found {len(fields)} fields in your Clio database:\n")
    for field in fields:
        print(f"- {field.get('name', 'Unknown')} : {field.get('id')}")
else:
    print("❌ Failed to fetch fields:")
    print(response.text)

Fetching all Custom Fields...

✅ Found 7 fields in your Clio database:

- Accident Date : 482522
- At-Fault Party : 482525
- Accident Description : 482528
- Accident Location : 483278
- Client Plate Number : 483281
- Statute of Limitations Date : 483284
- Injuries Reported : 483287


In [12]:
import os
import requests
import json
from dotenv import load_dotenv

load_dotenv()
ACCESS_TOKEN = os.getenv("CLIO_ACCESS_TOKEN")

MATTER_ID = 14537018  
TEMPLATE_ID = 359777  

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

print("--- TRIGGERING DOCUMENT AUTOMATION ---")
doc_url = "https://eu.app.clio.com/api/v4/document_automations.json"

# We added BOTH filename and formats as required by Clio!
doc_payload = {
    "data": {
        "matter": {"id": MATTER_ID},
        "document_template": {"id": TEMPLATE_ID},
        "filename": "Retainer_Agreement_Richards_Law", # Removed the .pdf extension to avoid double-naming
        "formats": ["pdf"] 
    }
}

print(f"Sending payload:\n{json.dumps(doc_payload, indent=2)}\n")

doc_res = requests.post(doc_url, headers=headers, json=doc_payload)

print(f"Status Code: {doc_res.status_code}")
if doc_res.status_code in [200, 201]:
    print("✅ SUCCESS! Document generated.")
    print(json.dumps(doc_res.json(), indent=2))
else:
    print("❌ FAILED!")
    print(doc_res.text)

--- TRIGGERING DOCUMENT AUTOMATION ---
Sending payload:
{
  "data": {
    "matter": {
      "id": 14537018
    },
    "document_template": {
      "id": 359777
    },
    "filename": "Retainer_Agreement_Richards_Law",
    "formats": [
      "pdf"
    ]
  }
}

Status Code: 201
✅ SUCCESS! Document generated.
{
  "data": {
    "id": 12985145,
    "etag": "\"c6233969a343366a68f6d0d2c3633492\""
  }
}


In [13]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
ACCESS_TOKEN = os.getenv("CLIO_ACCESS_TOKEN")

# Fetching all templates to see the new ID
url = "https://eu.app.clio.com/api/v4/document_templates.json"
headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    templates = response.json().get("data", [])
    print(f"✅ Found {len(templates)} templates in your account:")
    for t in templates:
        # Some versions of the API use 'name', others use 'filename'
        name = t.get('filename') or t.get('name') or "Unnamed Template"
        print(f"ID: {t['id']} | Name: {name}")
else:
    print(f"❌ Error: {response.status_code}")
    print(response.text)
    

✅ Found 1 templates in your account:
ID: 359795 | Name: Unnamed Template


In [1]:
import os
import requests
import json
from dotenv import load_dotenv

load_dotenv()
ACCESS_TOKEN = os.getenv("CLIO_ACCESS_TOKEN")
headers = {"Authorization": f"Bearer {ACCESS_TOKEN}", "Content-Type": "application/json"}

MATTER_ID = 14537018 # <-- REPLACE WITH YOUR SELECTED MATTER ID

print("--- DEBUGGING RESPONSIBLE ATTORNEY ---")
matter_url = f"https://eu.app.clio.com/api/v4/matters/{MATTER_ID}.json?fields=id,responsible_attorney{{id}}"
res = requests.get(matter_url, headers=headers)

if res.status_code == 200:
    data = res.json().get("data", {})
    attorney = data.get("responsible_attorney")
    print(f"Raw Attorney Data: {attorney}")
    if attorney is None:
        print("🚨 BINGO: The responsible_attorney is null! You need to assign an attorney to this Matter in Clio.")
    else:
        print(f"✅ Attorney ID is: {attorney.get('id')}")
else:
    print(f"Error fetching matter: {res.text}")

--- DEBUGGING RESPONSIBLE ATTORNEY ---
Raw Attorney Data: None
🚨 BINGO: The responsible_attorney is null! You need to assign an attorney to this Matter in Clio.


In [3]:
import os
import time
import requests
import json
from dotenv import load_dotenv

load_dotenv()
ACCESS_TOKEN = os.getenv("CLIO_ACCESS_TOKEN")
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

# Update this if you are using a different Matter ID for testing!
MATTER_ID = 14537018 
TEMPLATE_ID = 359795 # The template ID from your app.py

print("--- 1. TRIGGERING DOCUMENT AUTOMATION ---")
doc_url = "https://eu.app.clio.com/api/v4/document_automations.json"
doc_payload = {
    "data": {
        "matter": {"id": MATTER_ID},
        "document_template": {"id": TEMPLATE_ID},
        "filename": "Debug_Retainer_Agreement",
        "formats": ["pdf"]
    }
}

doc_res = requests.post(doc_url, headers=headers, json=doc_payload)

if doc_res.status_code in [200, 201]:
    doc_automation_id = doc_res.json()["data"]["id"]
    print(f"✅ Triggered! Automation ID: {doc_automation_id}")
    
    print("\n--- 2. POLLING FOR STATUS ---")
    # REMOVED the ?fields= parameter entirely
    poll_url = f"https://eu.app.clio.com/api/v4/document_automations/{doc_automation_id}.json"
    
    # Let's try 15 times (30 seconds) to give it extra time
    for attempt in range(15): 
        time.sleep(2)
        poll_res = requests.get(poll_url, headers=headers)
        
        if poll_res.status_code == 200:
            raw_data = poll_res.json()
            auto_data = raw_data.get("data", {})
            status = auto_data.get("status")
            
            print(f"Attempt {attempt + 1} | Status: {status} | Raw Data: {json.dumps(auto_data)}")
            
            if status == "complete":
                print("\n🎉 SUCCESS! Document generated.")
                break
            elif status == "failed":
                print("\n❌ FAILED! Clio encountered an error generating the document.")
                break
        else:
            print(f"Error polling: {poll_res.status_code} - {poll_res.text}")
else:
    print(f"❌ Failed to trigger automation: {doc_res.text}")

--- 1. TRIGGERING DOCUMENT AUTOMATION ---
✅ Triggered! Automation ID: 12988346

--- 2. POLLING FOR STATUS ---
Attempt 1 | Status: None | Raw Data: {"id": 12988346, "etag": "\"c62fba69a48f566a68f6d0d2c3633492\""}
Attempt 2 | Status: None | Raw Data: {"id": 12988346, "etag": "\"c62fba69a48f566a68f6d0d2c3633492\""}
Attempt 3 | Status: None | Raw Data: {"id": 12988346, "etag": "\"c62fba69a48f566a68f6d0d2c3633492\""}
Attempt 4 | Status: None | Raw Data: {"id": 12988346, "etag": "\"c62fba69a48f566a68f6d0d2c3633492\""}
Attempt 5 | Status: None | Raw Data: {"id": 12988346, "etag": "\"c62fba69a48f646a68f6d0d2c3633492\""}
Attempt 6 | Status: None | Raw Data: {"id": 12988346, "etag": "\"c62fba69a48f646a68f6d0d2c3633492\""}
Attempt 7 | Status: None | Raw Data: {"id": 12988346, "etag": "\"c62fba69a48f646a68f6d0d2c3633492\""}
Attempt 8 | Status: None | Raw Data: {"id": 12988346, "etag": "\"c62fba69a48f646a68f6d0d2c3633492\""}
Attempt 9 | Status: None | Raw Data: {"id": 12988346, "etag": "\"c62fba69a

In [4]:
import os
import time
import requests
import json
from dotenv import load_dotenv

load_dotenv()
ACCESS_TOKEN = os.getenv("CLIO_ACCESS_TOKEN")
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

# Update this if you are using a different Matter ID for testing!
MATTER_ID = 14537018 
TEMPLATE_ID = 359795 
EXPECTED_FILENAME = "Debug_Retainer_Agreement_Test"

print("--- 1. TRIGGERING DOCUMENT AUTOMATION ---")
doc_url = "https://eu.app.clio.com/api/v4/document_automations.json"
doc_payload = {
    "data": {
        "matter": {"id": MATTER_ID},
        "document_template": {"id": TEMPLATE_ID},
        "filename": EXPECTED_FILENAME,
        "formats": ["pdf"]
    }
}

doc_res = requests.post(doc_url, headers=headers, json=doc_payload)

if doc_res.status_code in [200, 201]:
    print(f"✅ Triggered! Automation ID: {doc_res.json()['data']['id']}")
    
    print(f"\n--- 2. POLLING MATTER {MATTER_ID} DOCUMENTS ---")
    
    document_id = None
    # We ask Clio for all documents inside this specific Matter
    docs_url = f"https://eu.app.clio.com/api/v4/documents.json?matter_id={MATTER_ID}&fields=id,name,filename"
    
    for attempt in range(15): # Try for up to 30 seconds
        time.sleep(2)
        docs_res = requests.get(docs_url, headers=headers)
        
        if docs_res.status_code == 200:
            docs_data = docs_res.json().get("data", [])
            print(f"Attempt {attempt + 1}: Found {len(docs_data)} documents in this Matter.")
            
            for doc in docs_data:
                # Clio might use 'name' or 'filename' depending on the exact object type
                doc_name = doc.get("filename") or doc.get("name") or ""
                
                # Check if our expected prefix is in the document name
                if EXPECTED_FILENAME in doc_name:
                    document_id = doc["id"]
                    print(f"\n🎉 SUCCESS! Found our generated PDF!")
                    print(f"Document ID: {document_id} | Name: {doc_name}")
                    break
            
            if document_id:
                break # Exit the polling loop if we found it!
        else:
            print(f"Error checking documents: {docs_res.status_code} - {docs_res.text}")
            
    if not document_id:
        print("\n❌ FAILED! Timed out. The document never appeared in the Matter.")
else:
    print(f"❌ Failed to trigger automation: {doc_res.text}")

--- 1. TRIGGERING DOCUMENT AUTOMATION ---
✅ Triggered! Automation ID: 12988373

--- 2. POLLING MATTER 14537018 DOCUMENTS ---
Attempt 1: Found 7 documents in this Matter.
Attempt 2: Found 7 documents in this Matter.
Attempt 3: Found 7 documents in this Matter.
Attempt 4: Found 7 documents in this Matter.
Attempt 5: Found 8 documents in this Matter.

🎉 SUCCESS! Found our generated PDF!
Document ID: 777423116 | Name: Debug_Retainer_Agreement_Test.pdf


In [5]:
import os
import requests
import smtplib
from email.message import EmailMessage
from dotenv import load_dotenv

load_dotenv()
ACCESS_TOKEN = os.getenv("CLIO_ACCESS_TOKEN")
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

# The Document ID we just successfully generated in the previous cell
DOCUMENT_ID = 777423116 

print(f"--- 1. DOWNLOADING DOCUMENT {DOCUMENT_ID} FROM CLIO ---")
download_url = f"https://eu.app.clio.com/api/v4/documents/{DOCUMENT_ID}/download"
pdf_res = requests.get(download_url, headers=headers)

if pdf_res.status_code == 200:
    pdf_bytes = pdf_res.content
    print("✅ PDF downloaded successfully! Size:", len(pdf_bytes), "bytes")
    
    print("\n--- 2. CONSTRUCTING AND SENDING EMAIL ---")
    msg = EmailMessage()
    msg['Subject'] = "Notebook Test: Your Case & Retainer Agreement"
    
    # Using your SMTP_USER from the .env file as the sender
    msg['From'] = os.getenv("SMTP_USER") 
    
    # Sending to your test university email
    msg['To'] = "19774907@sun.ac.za" 
    
    email_body = """Hello!
    
This is a test email sent directly from the Jupyter Notebook to verify the SMTP settings and PDF attachment functionality.

If you are reading this and there is a PDF attached, the automation is ready to be fully integrated into the Streamlit app!

Best regards,
Your Legal Engineer App
"""
    msg.set_content(email_body)
    
    # Attach the downloaded PDF
    msg.add_attachment(
        pdf_bytes, 
        maintype='application', 
        subtype='pdf', 
        filename="Debug_Retainer_Agreement_Test.pdf"
    )

    try:
        # Fetch SMTP credentials from .env
        smtp_server = os.getenv("SMTP_SERVER", "smtp.gmail.com")
        smtp_port = int(os.getenv("SMTP_PORT", 587))
        smtp_user = os.getenv("SMTP_USER")
        smtp_pass = os.getenv("SMTP_PASSWORD")

        print("Connecting to SMTP server...")
        with smtplib.SMTP(smtp_server, smtp_port) as server:
            server.starttls()
            server.login(smtp_user, smtp_pass)
            server.send_message(msg)
            
        print("🎉 SUCCESS! The test email has been sent. Check your 19774907@sun.ac.za inbox!")
    except Exception as e:
        print(f"\n❌ FAILED to send email! Error details: {e}")
        print("Double-check your SMTP_PASSWORD (the 16-digit Google App Password) in your .env file.")
else:
    print(f"❌ Failed to download PDF from Clio. Status Code: {pdf_res.status_code}")
    print(pdf_res.text)

--- 1. DOWNLOADING DOCUMENT 777423116 FROM CLIO ---
✅ PDF downloaded successfully! Size: 107932 bytes

--- 2. CONSTRUCTING AND SENDING EMAIL ---
Connecting to SMTP server...
🎉 SUCCESS! The test email has been sent. Check your 19774907@sun.ac.za inbox!


In [14]:
import os
import requests
import json
from dotenv import load_dotenv

load_dotenv()
ACCESS_TOKEN = os.getenv("CLIO_ACCESS_TOKEN")
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

print("--- SPYING ON CALENDAR OWNER EXACT STRUCTURE ---")
# We ask for the calendar_owner exactly, avoiding the forbidden 'calendar' field
entries_url = "https://eu.app.clio.com/api/v4/calendar_entries.json?fields=id,summary,calendar_owner{id,type}"
entries_res = requests.get(entries_url, headers=headers)

if entries_res.status_code == 200:
    entries = entries_res.json().get("data", [])
    if entries:
        print("✅ Found your test entry! Here is the exact 'calendar_owner' format:")
        print(json.dumps(entries[0].get("calendar_owner"), indent=2))
        
        print("\n--- TEST: CREATING ENTRY WITH SPY DATA ---")
        calendar_url = "https://eu.app.clio.com/api/v4/calendar_entries.json"
        
        calendar_payload = {
            "data": {
                "summary": "Statute of Limitations - SPY TEST",
                "start_at": "2032-05-05T09:00:00Z",
                "end_at": "2032-05-05T10:00:00Z",
                "matter": {"id": 14537018},
                # Plugging the exact spied object right back in!
                "calendar_owner": entries[0].get("calendar_owner")
            }
        }
        
        entry_res = requests.post(calendar_url, headers=headers, json=calendar_payload)
        
        if entry_res.status_code in [200, 201]:
            print("🎉 SUCCESS! Calendar entry created.")
        else:
            print(f"❌ FAILED! Status Code: {entry_res.status_code}")
            print(json.dumps(entry_res.json(), indent=2))
    else:
        print("🚨 No entries found! Please make sure you saved a test event in Clio.")
else:
    print(f"Error fetching entries: {entries_res.text}")

--- SPYING ON CALENDAR OWNER EXACT STRUCTURE ---
✅ Found your test entry! Here is the exact 'calendar_owner' format:
{
  "id": 437717,
  "type": "UserCalendar"
}

--- TEST: CREATING ENTRY WITH SPY DATA ---
🎉 SUCCESS! Calendar entry created.
